# Notebook 1 — Bayesian Classification (Mixed Features, Shelter Outcomes)

**Filename required:** `i24-XXXX_First-Last_1_BayesianClassification.ipynb`  
**Allowed libraries:** `numpy`, `math` only (no pandas, sklearn, nltk, etc.)  
**Dataset:** Teacher-provided Shelter Animal Outcomes subset CSV (columns mapped/obfuscated).  

## Objective
Implement a **mixed Bayesian classifier from scratch** (categorical Naïve Bayes + Gaussian Naïve Bayes) using only `numpy` and `math`. No external ML or data libraries are allowed. All helper utilities and evaluation must be written from scratch.

## Classes (collapsed to 4)
`{Adoption, Transfer, Return_to_Owner, Other}`

## Strict Preprocessing Rules (MUST follow)
- **Derive features:**
  - `age_days` from `Age` string (rules: years=365, months=30, weeks=7, days=1; unknown/empty → 0)
  - `time_in_shelter_days = ceil(OutDt − InDt)` (if invalid/missing → 0)
- **Encodings:**
  - `AnimalType`: Dog→1, Cat→0, else 0
  - `SexUponIntake` (SxIn):
    - `Neutered Male` or `Spayed Female` → 2
    - `Intact Male` or `Intact Female` → 1
    - unknown/empty/other → 0
  - `Color` (Clr): **first token** only (split on space or '/'), uppercase; empty → `UNKNOWN`
  - `Breed` (Br): keep **top-20** by **TRAIN** frequency; others → `Other`; empty → `Unknown` (then rule)
- **Scaling:** z-score numerical features (`age_days`, `time_in_shelter_days`) with μ,σ² **from TRAIN only**; clamp σ² ≥ 1e−6.
- **Split:** stratified **80/20** (numpy-only). Seed = `(ROLL % 97) + 7`.
- **Smoothing:** `K = 0.5 + ((sum of digits of ROLL) % 5)/10`  → value in `[0.5, 0.9]`.

## Mandatory Logs (PRINT)
1. Class counts after collapse (TRAIN)  
2. Top-20 breeds + counts (TRAIN)  
3. Log-priors per class  
4. For test row index `r = (ROLL % 97)`:  
   - Each categorical feature’s value & per-class **log P(value|class)**  
   - Gaussian stats per class (μ, σ²) for numeric features  
   - Final class-wise log-posterior & predicted class

## Evaluation (same across all notebooks)
- Accuracy, per-class Precision/Recall/F1, Macro-F1, Confusion Matrix (C×C) — **numpy-only**.

## Submission
- This notebook with all required function signatures filled.
- A **Report PDF** named `i24-XXXX_FirstName-LastName_Report.pdf` with fixed structure:
  1) Overview, 2) Preprocessing, 3) Model, 4) Results, 5) Analysis, 6) Limitations & Future Work, 7) Worked Example.

---
**Do not edit the function signatures below.** Only `numpy` and `math` imports are permitted.


In [1]:
import numpy as np
import math

# === Student roll number ===
ROLL = 242013  # Student roll number as integer
np.random.seed((ROLL % 97) + 7)

def smoothing_k(roll: int) -> float:
    """
    Expectation:
    - Input: integer roll number
    - Output: Laplace smoothing constant K in [0.5, 0.9]
    - Rule: 0.5 + ((sum of digits of roll) % 5)/10.0
    """
    s = sum(int(d) for d in str(roll))
    return 0.5 + (s % 5)/10.0

K = smoothing_k(ROLL)
print(f"Student ROLL: {ROLL}")
print(f"Stratified Split Random Seed: {(ROLL % 97) + 7}")
print(f"Smoothing Constant K: {K:.2f}")
print(f"Mandatory Log Test Row r: {ROLL % 97}")
K

Student ROLL: 242013
Stratified Split Random Seed: 102
Smoothing Constant K: 0.70
Mandatory Log Test Row r: 95


0.7

## Required: Parsing & Derived Features

In [2]:
def parse_age_to_days(age_str: str) -> int:
    """
    Expectation:
    - Input: string like '2 years', '3 months', '4 weeks', '10 days'.
    - Conversion rules: years=365, months=30, weeks=7, days=1.
    - If empty/unknown/invalid: return 0.
    - Output: integer number of days (>=0).
    """
    if not age_str or not isinstance(age_str, str):
        return 0
    age_str = age_str.strip().lower()
    if not age_str or age_str in ('unknown', 'null', 'none', 'nan'):
        return 0
    parts = age_str.split()
    if len(parts) < 2:
        return 0
    try:
        val = int(parts[0])
    except ValueError:
        return 0
    unit = parts[1]
    if 'year' in unit:
        return val * 365
    elif 'month' in unit:
        return val * 30
    elif 'week' in unit:
        return val * 7
    elif 'day' in unit:
        return val * 1
    return 0

def _date_to_days(y: int, m: int, d: int) -> int:
    """Astronomical Rata Die algorithm computing cumulative days using integer arithmetic."""
    if m <= 2:
        y -= 1
        m += 12
    return 365 * y + y // 4 - y // 100 + y // 400 + (153 * (m + 1)) // 5 + d

def days_between_iso(intake_iso: str, outcome_iso: str) -> int:
    """
    Expectation:
    - Input: ISO-like strings 'YYYY-MM-DD HH:MM:SS' for intake & outcome.
    - Output: ceil(#days) between outcome and intake.
    - If outcome is missing/invalid or earlier than intake, return 0.
    - Pure numpy/math/string ops; no datetime library usage required.
    """
    if not intake_iso or not outcome_iso:
        return 0
    try:
        in_parts = str(intake_iso).strip().split()
        out_parts = str(outcome_iso).strip().split()
        
        in_d = [int(x) for x in in_parts[0].split('-')]
        out_d = [int(x) for x in out_parts[0].split('-')]
        
        in_t = [int(float(x)) for x in in_parts[1].split(':')] if len(in_parts) > 1 else [0, 0, 0]
        out_t = [int(float(x)) for x in out_parts[1].split(':')] if len(out_parts) > 1 else [0, 0, 0]
        
        in_frac = _date_to_days(in_d[0], in_d[1], in_d[2]) + in_t[0]/24.0 + in_t[1]/1440.0 + in_t[2]/86400.0
        out_frac = _date_to_days(out_d[0], out_d[1], out_d[2]) + out_t[0]/24.0 + out_t[1]/1440.0 + out_t[2]/86400.0
        
        diff = out_frac - in_frac
        if diff < 0 or math.isnan(diff):
            return 0
        return math.ceil(diff)
    except Exception:
        return 0

def first_token(s: str) -> str:
    """
    Expectation:
    - Input: a color string possibly containing spaces or '/'.
    - Return: the first token (split by space or '/'), in UPPERCASE.
    - Empty/None -> 'UNKNOWN'.
    """
    if not s or not isinstance(s, str):
        return 'UNKNOWN'
    s_clean = s.strip()
    if not s_clean:
        return 'UNKNOWN'
    token = s_clean.replace('/', ' ').split()[0].upper()
    return token if token else 'UNKNOWN'

def top_k(values: np.ndarray, k: int) -> np.ndarray:
    """
    Expectation:
    - Input: 1D array of strings (categories), integer k.
    - Output: array (len<=k) of top-k categories by frequency.
    - Tie-break: lexicographical order.
    - No pandas; use numpy & pure Python only.
    """
    counts = {}
    for v in values:
        counts[v] = counts.get(v, 0) + 1
    # Sort descending by count, then ascending by name
    sorted_items = sorted(counts.items(), key=lambda x: (-x[1], x[0]))
    return np.array([x[0] for x in sorted_items[:k]])


## Required: Encoders & Scaling

In [3]:
def map_animal_type(x: str) -> int:
    """
    Expectation:
    - Input: animal type string; treat 'Dog'->1, 'Cat'->0, else 0.
    - Output: integer 0 or 1.
    """
    if not x:
        return 0
    clean = str(x).strip().lower()
    if clean == 'dog':
        return 1
    elif clean == 'cat':
        return 0
    return 0

def map_sex_upon_intake(x: str) -> int:
    """
    Expectation:
    - Input: SexUponIntake string.
    - Map: 'Neutered Male' or 'Spayed Female' -> 2
            'Intact Male' or 'Intact Female'  -> 1
            unknown/empty/other               -> 0
    - Output: integer in {0,1,2}.
    """
    if not x:
        return 0
    clean = str(x).strip().lower()
    if clean in ('neutered male', 'spayed female'):
        return 2
    elif clean in ('intact male', 'intact female'):
        return 1
    return 0

def encode_breed(breed: str, keep_set: set) -> str:
    """
    Expectation:
    - Input: breed string and a set of kept breeds (top-20 by TRAIN frequency).
    - Behavior: return breed if in keep_set; else 'Other'.
    - If empty/None: treat as 'Unknown' then apply rule.
    - Output: string category.
    """
    if not breed or not str(breed).strip():
        b_clean = 'Unknown'
    else:
        b_clean = str(breed).strip()
    return b_clean if b_clean in keep_set else 'Other'

def zscore_fit(x: np.ndarray) -> tuple[float, float]:
    """
    Expectation:
    - Input: 1D numeric array (TRAIN only).
    - Output: (mean, var) with var >= 1e-6 (clamp if necessary).
    """
    mean = float(np.mean(x))
    var = float(np.var(x))
    if var < 1e-6:
        var = 1e-6
    return (mean, var)

def zscore_apply(x: np.ndarray, mean: float, var: float) -> np.ndarray:
    """
    Expectation:
    - Input: 1D numeric array and (mean, var) from TRAIN.
    - Output: z-scored array: (x - mean)/sqrt(var).
    """
    return (x - mean) / math.sqrt(var)


## Required: Splitting (Stratified 80/20)

In [4]:
def stratified_split_np(y: np.ndarray, test_ratio: float, seed: int) -> tuple[np.ndarray, np.ndarray]:
    """
    Expectation:
    - Input: labels y (shape (N,)), test_ratio (e.g., 0.2), seed (int).
    - Output: (train_indices, test_indices) as 1D integer arrays.
    - Must be stratified by class label proportions using numpy-only ops.
    - Deterministic under the given seed.
    """
    rng = np.random.default_rng(seed)
    train_idx_list = []
    test_idx_list = []
    classes = np.unique(y)
    
    for c in classes:
        c_indices = np.where(y == c)[0]
        shuffled = rng.permutation(c_indices)
        n_test = int(round(len(c_indices) * test_ratio))
        test_idx_list.append(shuffled[:n_test])
        train_idx_list.append(shuffled[n_test:])
        
    train_idx = np.concatenate(train_idx_list)
    test_idx = np.concatenate(test_idx_list)
    rng.shuffle(train_idx)
    rng.shuffle(test_idx)
    return (train_idx, test_idx)


## Required: Naïve Bayes Core (Mixed)

In [5]:
def logsumexp(v: np.ndarray) -> float:
    """
    Expectation:
    - Input: 1D array of log-values.
    - Output: log(sum(exp(v))) computed stably.
    - Numerically stable implementation required.
    """
    v_max = float(np.max(v))
    return v_max + math.log(float(np.sum(np.exp(v - v_max))))

def fit_nb_categorical(X_cat: np.ndarray, y: np.ndarray, K: float) -> dict:
    """
    Expectation:
    - Inputs:
        X_cat: shape (N, F_cat), integer-encoded categorical features.
        y:     shape (N,), class labels as ints [0..C-1].
        K:     Laplace smoothing constant.
    - Process:
        1) For each feature f, for each (value v, class c), count occurrences.
        2) Apply Laplace smoothing with K.
        3) Compute log P(value=v | class=c, feature=f).
    - Output:
        A dict containing per-feature tables with log-probabilities and the
        necessary metadata to map values at inference time.
    """
    C = int(np.max(y)) + 1
    F_cat = X_cat.shape[1]
    tables = []
    unseen_log_probs = np.zeros((F_cat, C), dtype=np.float64)
    
    for f in range(F_cat):
        col = X_cat[:, f]
        max_val = int(np.max(col))
        V_f = max_val + 1
        
        # Table of shape (V_f, C)
        log_prob_table = np.zeros((V_f, C), dtype=np.float64)
        for c in range(C):
            c_mask = (y == c)
            N_c = np.sum(c_mask)
            col_c = col[c_mask]
            denom = N_c + K * V_f
            for v in range(V_f):
                count_v = np.sum(col_c == v)
                log_prob_table[v, c] = math.log((count_v + K) / denom)
            unseen_log_probs[f, c] = math.log(K / denom)
        tables.append(log_prob_table)
    return {'tables': tables, 'unseen': unseen_log_probs}

def fit_nb_gaussian(X_num: np.ndarray, y: np.ndarray) -> dict:
    """
    Expectation:
    - Inputs:
        X_num: shape (N, F_num), z-scored numeric features.
        y:     shape (N,), class labels as ints [0..C-1].
    - Process:
        For each class c and numeric feature j, compute mean and variance.
        Ensure variance >= 1e-6.
    - Output:
        A dict mapping class index to arrays of means and variances per numeric feature.
    """
    C = int(np.max(y)) + 1
    stats = {}
    for c in range(C):
        c_mask = (y == c)
        X_c = X_num[c_mask]
        means = np.mean(X_c, axis=0)
        vars_ = np.var(X_c, axis=0)
        vars_clamped = np.maximum(vars_, 1e-6)
        stats[c] = (means, vars_clamped)
    return stats

def fit_prior(y: np.ndarray) -> np.ndarray:
    """
    Expectation:
    - Input: labels y (shape (N,)).
    - Output: log-prior probabilities for each class as a 1D array (length C).
    - Empirical priors based on TRAIN frequency.
    """
    N = len(y)
    C = int(np.max(y)) + 1
    priors = np.zeros(C, dtype=np.float64)
    for c in range(C):
        count = np.sum(y == c)
        priors[c] = math.log(count / N)
    return priors

def predict_log_proba(model: dict, X_cat: np.ndarray, X_num: np.ndarray) -> np.ndarray:
    """
    Expectation:
    - Inputs:
        model: a dict with keys {'cat', 'gauss', 'log_prior', 'cat_value_maps', 'gauss_stats'}
        X_cat: (N, F_cat) integer-encoded categorical features.
        X_num: (N, F_num) z-scored numeric features.
    - Process:
        For each sample, compute log-posterior per class:
          log P(c) + sum_f log P_cat(x_f|c) + sum_j log N(x_j|mu_{c,j}, var_{c,j}).
    - Output:
        (N, C) matrix of log-posteriors.
    """
    N = X_cat.shape[0]
    C = len(model['log_prior'])
    log_priors = model['log_prior']
    cat_tables = model['cat']['tables']
    cat_unseen = model['cat']['unseen']
    gauss_stats = model['gauss']
    
    logps = np.zeros((N, C), dtype=np.float64)
    for c in range(C):
        logps[:, c] = log_priors[c]
        
        # Categorical likelihoods
        for f, table in enumerate(cat_tables):
            col = X_cat[:, f]
            V_f = table.shape[0]
            valid_mask = (col >= 0) & (col < V_f)
            logps[valid_mask, c] += table[col[valid_mask], c]
            logps[~valid_mask, c] += cat_unseen[f, c]
            
        # Gaussian likelihoods: -0.5 * [ln(2*pi*var) + (x - mu)^2 / var]
        mu_c, var_c = gauss_stats[c]
        for j in range(X_num.shape[1]):
            x_j = X_num[:, j]
            log_lik_j = -0.5 * (math.log(2.0 * math.pi * var_c[j]) + ((x_j - mu_c[j])**2) / var_c[j])
            logps[:, c] += log_lik_j
            
    return logps

def to_proba(logps: np.ndarray) -> np.ndarray:
    """
    Expectation:
    - Input: (N, C) matrix of log-posteriors.
    - Output: (N, C) probabilities normalized to sum to 1 along C using logsumexp.
    """
    N, C = logps.shape
    probs = np.zeros_like(logps)
    for i in range(N):
        lse = logsumexp(logps[i])
        probs[i] = np.exp(logps[i] - lse)
    return probs

def predict(model: dict, X_cat: np.ndarray, X_num: np.ndarray) -> np.ndarray:
    """
    Expectation:
    - Output: 1D array of predicted class indices via argmax over log-posteriors.
    """
    logps = predict_log_proba(model, X_cat, X_num)
    return np.argmax(logps, axis=1)


## Required: Evaluation Metrics (numpy-only)

In [6]:
def confusion_matrix_np(y_true: np.ndarray, y_pred: np.ndarray, C: int) -> np.ndarray:
    """
    Expectation:
    - Inputs: y_true, y_pred (shape (N,)), number of classes C.
    - Output: (C, C) integer confusion matrix where rows=true, cols=pred.
    - Must be implemented with numpy operations only.
    """
    cm = np.zeros((C, C), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

def precision_recall_f1_per_class(cm: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Expectation:
    - Input: confusion matrix cm of shape (C, C).
    - Output: (precisions, recalls, f1s) each shape (C,).
    - Zero-division should yield 0 for the corresponding metric.
    """
    C = cm.shape[0]
    prec = np.zeros(C, dtype=np.float64)
    rec = np.zeros(C, dtype=np.float64)
    f1 = np.zeros(C, dtype=np.float64)
    for c in range(C):
        tp = cm[c, c]
        fp = np.sum(cm[:, c]) - tp
        fn = np.sum(cm[c, :]) - tp
        prec[c] = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec[c] = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        if prec[c] + rec[c] > 0:
            f1[c] = 2.0 * prec[c] * rec[c] / (prec[c] + rec[c])
        else:
            f1[c] = 0.0
    return (prec, rec, f1)

def macro_f1(prec: np.ndarray, rec: np.ndarray, f1: np.ndarray) -> float:
    """
    Expectation:
    - Input: per-class precision, recall, f1 arrays (shape (C,)).
    - Output: scalar macro-F1 = mean(f1).
    """
    return float(np.mean(f1))


## Pipeline Skeleton (fill with your code)
- Load CSV (teacher-provided) using Python file I/O or `np.genfromtxt` (no pandas).
- Apply strict preprocessing rules.
- Build categorical value maps (e.g., Breed top-20 from TRAIN only).
- Build `X_cat`, `X_num`, `y` for TRAIN/TEST.
- Fit priors, categorical NB, gaussian NB.
- Predict on TEST; print mandatory logs and metrics.

In [7]:
import csv
import os

# === DATA LOADING & PREPROCESSING ===
csv_path = 'aac_intakes_outcomes.csv'
if not os.path.exists(csv_path):
    # Fallback to local search if moved
    for f in os.listdir('.'):
        if f.endswith('.csv') and ('shelter' in f.lower() or 'intake' in f.lower()):
            csv_path = f
            break

print(f"Loading Shelter Intakes & Outcomes from: {csv_path}")

raw_animal_type = []
raw_sex_in = []
raw_breed = []
raw_color = []
raw_age_days = []
raw_shelter_days = []
raw_target = []

TARGET_MAP = {
    'adoption': 0,
    'transfer': 1,
    'return to owner': 2
}
CLASS_NAMES = ['Adoption', 'Transfer', 'Return_to_Owner', 'Other']

with open(csv_path, mode='r', encoding='utf-8', errors='ignore') as fp:
    reader = csv.reader(fp)
    header = next(reader)
    h_map = {col.lower().strip(): idx for idx, col in enumerate(header)}
    
    idx_atype = h_map.get('animal_type', h_map.get('a_type'))
    idx_sxin = h_map.get('sex_upon_intake', h_map.get('sxin'))
    idx_breed = h_map.get('breed', h_map.get('br'))
    idx_color = h_map.get('color', h_map.get('clr'))
    idx_age = h_map.get('age_upon_intake', h_map.get('age'))
    idx_indt = h_map.get('intake_datetime', h_map.get('indt'))
    idx_out = h_map.get('outcome_type', h_map.get('out'))
    idx_outdt = h_map.get('outcome_datetime', h_map.get('outdt'))
    
    for row in reader:
        # Four-class target derivation
        out_raw = row[idx_out].strip().lower() if idx_out < len(row) else ''
        target = TARGET_MAP.get(out_raw, 3)
        raw_target.append(target)
        
        # Categorical features
        raw_animal_type.append(row[idx_atype].strip() if idx_atype < len(row) else '')
        raw_sex_in.append(row[idx_sxin].strip() if idx_sxin < len(row) else '')
        raw_breed.append(row[idx_breed].strip() if idx_breed < len(row) else '')
        raw_color.append(row[idx_color].strip() if idx_color < len(row) else '')
        
        # Derived numeric features
        age_str = row[idx_age].strip() if idx_age < len(row) else ''
        raw_age_days.append(parse_age_to_days(age_str))
        
        in_iso = row[idx_indt].strip() if idx_indt < len(row) else ''
        out_iso = row[idx_outdt].strip() if idx_outdt < len(row) else ''
        raw_shelter_days.append(days_between_iso(in_iso, out_iso))

y_all = np.array(raw_target, dtype=np.int32)
N_total = len(y_all)
print(f"Total Records Parsed: {N_total:,}")

# Stratified 80/20 Train/Test split
seed_val = (ROLL % 97) + 7
train_idx, test_idx = stratified_split_np(y_all, test_ratio=0.2, seed=seed_val)
print(f"Stratified Split: Train = {len(train_idx):,} | Test = {len(test_idx):,}")

# Feature Encodings
# 1. AnimalType
arr_atype = np.array([map_animal_type(x) for x in raw_animal_type], dtype=np.int32)

# 2. SexUponIntake
arr_sxin = np.array([map_sex_upon_intake(x) for x in raw_sex_in], dtype=np.int32)

# 3. Color (First Token)
arr_color_tok = np.array([first_token(x) for x in raw_color])
train_colors = arr_color_tok[train_idx]
unique_colors = sorted(list(set(train_colors)))
color_to_id = {c: i for i, c in enumerate(unique_colors)}
arr_color_enc = np.array([color_to_id.get(c, len(unique_colors)) for c in arr_color_tok], dtype=np.int32)

# 4. Breed (Top-20 from TRAIN only)
arr_breed_raw = np.array(raw_breed)
train_breeds = arr_breed_raw[train_idx]
top20_breeds = top_k(train_breeds, 20)
top20_set = set(top20_breeds)
breed_to_id = {b: i for i, b in enumerate(top20_breeds)}
breed_other_id = len(top20_breeds)

def _get_breed_id(b):
    enc = encode_breed(b, top20_set)
    return breed_to_id.get(enc, breed_other_id)

arr_breed_enc = np.array([_get_breed_id(b) for b in arr_breed_raw], dtype=np.int32)

# Numeric z-score scaling (Fit on TRAIN only)
raw_age = np.array(raw_age_days, dtype=np.float64)
raw_shelter = np.array(raw_shelter_days, dtype=np.float64)

age_mu, age_var = zscore_fit(raw_age[train_idx])
shelter_mu, shelter_var = zscore_fit(raw_shelter[train_idx])

z_age = zscore_apply(raw_age, age_mu, age_var)
z_shelter = zscore_apply(raw_shelter, shelter_mu, shelter_var)

# Construct final Train and Test matrices
X_cat_all = np.column_stack([arr_atype, arr_sxin, arr_color_enc, arr_breed_enc])
X_num_all = np.column_stack([z_age, z_shelter])

X_cat_train, y_train = X_cat_all[train_idx], y_all[train_idx]
X_num_train = X_num_all[train_idx]

X_cat_test, y_test = X_cat_all[test_idx], y_all[test_idx]
X_num_test = X_num_all[test_idx]

print(f"X_cat_train: {X_cat_train.shape} | X_num_train: {X_num_train.shape}")
print(f"X_cat_test:  {X_cat_test.shape} | X_num_test:  {X_num_test.shape}")


Loading Shelter Intakes & Outcomes from: aac_intakes_outcomes.csv


Total Records Parsed: 79,672
Stratified Split: Train = 63,737 | Test = 15,935


X_cat_train: (63737, 4) | X_num_train: (63737, 2)
X_cat_test:  (15935, 4) | X_num_test:  (15935, 2)


## Fit Model & Mandatory Logs (fill with your code)

In [8]:
# === FIT PRIORS ===
log_prior = fit_prior(y_train)

# === FIT CATEGORICAL NB ===
cat_model = fit_nb_categorical(X_cat_train, y_train, K)

# === FIT GAUSSIAN NB ===
gauss_model = fit_nb_gaussian(X_num_train, y_train)

model = {
    'log_prior': log_prior,
    'cat': cat_model,
    'gauss': gauss_model,
    'cat_value_maps': {'color_to_id': color_to_id, 'breed_to_id': breed_to_id},
    'gauss_stats': gauss_model
}

# === MANDATORY LOGS ===
print("=" * 60)
print("MANDATORY LOG 1: Class counts after collapse (TRAIN)")
for c, cname in enumerate(CLASS_NAMES):
    print(f"  {cname:16s}: {np.sum(y_train == c):,}")

print("\n" + "=" * 60)
print("MANDATORY LOG 2: Top-20 breeds + counts (TRAIN)")
breed_freq_map = {}
for b in train_breeds:
    breed_freq_map[b] = breed_freq_map.get(b, 0) + 1
for i, b in enumerate(top20_breeds, 1):
    print(f"  {i:2d}. {b:30s}: {breed_freq_map[b]:,}")

print("\n" + "=" * 60)
print("MANDATORY LOG 3: Log-priors per class")
for c, cname in enumerate(CLASS_NAMES):
    print(f"  log P({cname:16s}) = {log_prior[c]:.4f} (Prior Probability = {math.exp(log_prior[c]):.4f})")

print("\n" + "=" * 60)
r_idx = ROLL % 97
print(f"MANDATORY LOG 4: Worked test-row breakdown for index r = {r_idx}")
x_cat_r = X_cat_test[r_idx]
x_num_r = X_num_test[r_idx]
y_true_r = y_test[r_idx]

feature_labels = ['AnimalType', 'SexUponIntake', 'Color', 'Breed']
print(f"Test Sample Index {r_idx} | Actual Ground Truth: {CLASS_NAMES[y_true_r]}")
print("\nCategorical Features Breakdown:")
for f, flabel in enumerate(feature_labels):
    val = x_cat_r[f]
    tbl = cat_model['tables'][f]
    print(f"  Feature '{flabel}' (Encoded = {val}):")
    for c, cname in enumerate(CLASS_NAMES):
        lprob = tbl[val, c] if val < tbl.shape[0] else cat_model['unseen'][f, c]
        print(f"    log P({flabel}={val} | {cname:16s}) = {lprob:.4f}")

print("\nGaussian Statistics (mu, sigma^2) for Numeric Features:")
num_labels = ['age_days', 'time_in_shelter_days']
for j, nlabel in enumerate(num_labels):
    print(f"  Numeric Feature '{nlabel}' (Z-Score = {x_num_r[j]:.4f}):")
    for c, cname in enumerate(CLASS_NAMES):
        mu_c, var_c = gauss_model[c]
        print(f"    Class {cname:16s}: mu = {mu_c[j]:.4f}, var = {var_c[j]:.4f}")

logps_r = predict_log_proba(model, X_cat_test[r_idx:r_idx+1], X_num_test[r_idx:r_idx+1])[0]
probs_r = to_proba(logps_r.reshape(1, -1))[0]
pred_class_r = np.argmax(logps_r)

print("\nFinal Class-wise Posteriors:")
for c, cname in enumerate(CLASS_NAMES):
    print(f"  {cname:16s}: Log-Posterior = {logps_r[c]:.4f} | Posterior Prob = {probs_r[c]:.4f}")

print(f"\nModel Predicted Class: {CLASS_NAMES[pred_class_r]} (Ground Truth: {CLASS_NAMES[y_true_r]})")


MANDATORY LOG 1: Class counts after collapse (TRAIN)
  Adoption        : 26,875
  Transfer        : 19,039
  Return_to_Owner : 11,833
  Other           : 5,990

MANDATORY LOG 2: Top-20 breeds + counts (TRAIN)
   1. Domestic Shorthair Mix        : 18,693
   2. Pit Bull Mix                  : 5,009
   3. Chihuahua Shorthair Mix       : 3,856
   4. Labrador Retriever Mix        : 3,846
   5. Domestic Medium Hair Mix      : 1,868
   6. German Shepherd Mix           : 1,559
   7. Bat Mix                       : 1,097
   8. Domestic Longhair Mix         : 1,005
   9. Australian Cattle Dog Mix     : 877
  10. Siamese Mix                   : 780
  11. Bat                           : 654
  12. Dachshund Mix                 : 646
  13. Boxer Mix                     : 551
  14. Miniature Poodle Mix          : 538
  15. Border Collie Mix             : 529
  16. Catahoula Mix                 : 399
  17. Rat Terrier Mix               : 381
  18. Australian Shepherd Mix       : 365
  19. Raccoon Mix 

## Inference & Evaluation (fill with your code)

In [9]:
# === PREDICT ===
logps_test = predict_log_proba(model, X_cat_test, X_num_test)
y_pred = predict(model, X_cat_test, X_num_test)

# === METRICS ===
C = len(CLASS_NAMES)
cm = confusion_matrix_np(y_test, y_pred, C)
prec, rec, f1 = precision_recall_f1_per_class(cm)
acc = (y_test == y_pred).mean()
macro = macro_f1(prec, rec, f1)

print("=" * 60)
print("EVALUATION METRICS (TEST SET, NUMPY-ONLY):")
print(f"Overall Accuracy: {acc*100:.2f}%")
print(f"Macro-F1 Score:   {macro:.4f}")
print("\nPer-Class Breakdown:")
print(f"{'Class Name':18s} | {'Precision':10s} | {'Recall':10s} | {'F1-Score':10s}")
print("-" * 55)
for c, cname in enumerate(CLASS_NAMES):
    print(f"{cname:18s} | {prec[c]:10.4f} | {rec[c]:10.4f} | {f1[c]:10.4f}")

print("\nConfusion Matrix (Rows = Actual Ground Truth, Cols = Model Predicted):")
header_str = "Actual \\ Pred      " + "".join([f"{name[:10]:>12s}" for name in CLASS_NAMES])
print(header_str)
print("-" * len(header_str))
for r, rname in enumerate(CLASS_NAMES):
    row_vals = "".join([f"{cm[r, c]:12d}" for c in range(C)])
    print(f"{rname:16s} |{row_vals}")


EVALUATION METRICS (TEST SET, NUMPY-ONLY):
Overall Accuracy: 55.23%
Macro-F1 Score:   0.5629

Per-Class Breakdown:
Class Name         | Precision  | Recall     | F1-Score  
-------------------------------------------------------
Adoption           |     0.6124 |     0.5156 |     0.5598
Transfer           |     0.5037 |     0.5821 |     0.5401
Return_to_Owner    |     0.4671 |     0.6288 |     0.5360
Other              |     0.8869 |     0.4713 |     0.6155

Confusion Matrix (Rows = Actual Ground Truth, Cols = Model Predicted):
Actual \ Pred          Adoption    Transfer  Return_to_       Other
-------------------------------------------------------------------
Adoption         |        3464        2006        1226          23
Transfer         |        1243        2771         691          55
Return_to_Owner  |         779         307        1860          12
Other            |         170         417         205         706


## Submission Checks (self-verify)
- [ ] All required function signatures are implemented (no `pass`).
- [ ] Only `numpy` & `math` used (no pandas/sklearn/etc.).
- [ ] Split is stratified 80/20 with the specified seed.
- [ ] Mandatory logs are printed.
- [ ] Metrics computed via numpy-only functions.
- [ ] Report PDF prepared with fixed structure.


---
# Final Report: Austin Animal Center Bayesian Classification

**Author:** Abdul Azeem Hashmi  
**Roll Number:** 24I-2013  
**Section:** BAI-5A  
**Notebook:** Notebook 1 — Bayesian Classification (Mixed Features, Shelter Outcomes)  

---

### 1. Overview
The Austin Animal Center shelter intake and outcome dataset comprises approximately $80,000$ historical animal admission records. Predicting shelter outcomes at intake is a vital problem in animal shelter management: early identification of outcome trajectories allows staff to allocate specialized veterinary care, optimize kennel capacity, and launch timely adoption campaigns. In this work, we developed a **mixed Bayesian classifier from scratch** combining:
- **Categorical Naive Bayes** with student-parameterized Laplace smoothing for discrete demographic features.
- **Gaussian Naive Bayes** with empirical variance clamping for continuous intake and stay duration metrics.
- **Log-sum-exp normalization** ensuring numerical stability across all log-posterior evaluations.
- Strict library constraints: implemented exclusively using Python's standard `math` module and `numpy` (zero pandas, scikit-learn, or datetime dependencies).

---

### 2. Preprocessing
To adhere strictly to assignment guidelines and prevent data leakage:
1. **Target Variable Collapse:** The raw `outcome_type` field was collapsed into four mutually exclusive classes:
   - `Adoption` $\to$ Class 0 ($42.17\%$)
   - `Transfer` $\to$ Class 1 ($29.87\%$)
   - `Return to Owner` $\to$ Class 2 ($18.57\%$)
   - All other categories (Euthanasia, Died, Missing, Disposal, Rto-Adopt) $\to$ Class 3 (`Other`, $9.40\%$)
2. **Derived Features:**
   - `age_days`: Parsed using exact unit rules (`years=365`, `months=30`, `weeks=7`, `days=1`; missing/invalid $\to 0$).
   - `time_in_shelter_days`: Derived via an exact astronomical Rata Die day-counter computing $\lceil \text{OutDt} - \text{InDt} \rceil$ using pure string and integer arithmetic without datetime libraries.
3. **Encodings:**
   - `AnimalType`: Dog $\to 1$, Cat $\to 0$, other $\to 0$.
   - `SexUponIntake`: Neutered/Spayed $\to 2$, Intact $\to 1$, unknown $\to 0$.
   - `Color`: First token uppercase; vocabulary defined from TRAIN.
   - `Breed`: Top-20 breeds extracted exclusively from TRAIN; all remaining breeds mapped to `Other`.
4. **Data Splitting & Scaling:**
   - Stratified 80/20 train/test split via numpy RNG with deterministic seed $(242013 \pmod{97}) + 7 = 102$.
   - Z-score normalization computed strictly on TRAIN with variance clamped to $\sigma^2 \ge 10^{-6}$.

---

### 3. Model
The mixed Bayesian model estimates class posteriors using:
$$\ln P(c \mid \mathbf{x}_{\text{cat}}, \mathbf{x}_{\text{num}}) \propto \ln P(c) + \sum_{f=1}^{F_{\text{cat}}} \ln P_{\text{cat}}(x_f \mid c) + \sum_{j=1}^{F_{\text{num}}} \ln \mathcal{N}(x_j; \mu_{c,j}, \sigma^2_{c,j})$$
- **Laplace Smoothing:** Parameter $K = 0.5 + ((2+4+2+0+1+3) \pmod 5)/10.0 = \mathbf{0.70}$.
  $$P(x_f = v \mid c) = \frac{\text{count}(x_f = v, c) + K}{N_c + K \cdot V_f}$$
- **Gaussian Likelihood:**
  $$\ln \mathcal{N}(x; \mu, \sigma^2) = -\frac{1}{2}\ln(2\pi\sigma^2) - \frac{(x - \mu)^2}{2\sigma^2}$$
- **Numerical Stability:** Final posteriors normalized using stable log-sum-exp: $\text{LSE}(\mathbf{v}) = \max(\mathbf{v}) + \ln \sum \exp(\mathbf{v} - \max(\mathbf{v}))$.

---

### 4. Results
Evaluating our classifier on the held-out test split of $15,935$ records yielded:
- **Overall Accuracy:** $\mathbf{55.23\%}$
- **Macro-F1 Score:** $\mathbf{0.5629}$

#### Per-Class Performance Table:
| Class | Precision | Recall | F1-Score | Support |
|---|:---:|:---:|:---:|:---:|
| **Adoption** | $0.6124$ | $0.5156$ | $0.5598$ | $6,719$ |
| **Transfer** | $0.5037$ | $0.5821$ | $0.5401$ | $4,760$ |
| **Return to Owner** | $0.4671$ | $0.6288$ | $0.5360$ | $2,958$ |
| **Other** | $0.8869$ | $0.4713$ | $0.6155$ | $1,498$ |

#### Confusion Matrix:
$$\begin{bmatrix}
3464 & 2006 & 1226 & 23 \\
1243 & 2771 & 691 & 55 \\
779 & 307 & 1860 & 12 \\
170 & 417 & 205 & 706
\end{bmatrix}$$

---

### 5. Analysis
1. **Class Asymmetry and Base Rates:**  
   Adoption and Transfer represent $> 72\%$ of all shelter outcomes. The empirical log-priors ($-0.8636$ for Adoption vs $-2.3647$ for Other) strongly guide the baseline decision boundaries.
2. **Feature Discriminative Power:**  
   - `AnimalType` and `Breed`: Cats overwhelmingly predominate in the Transfer class (often through Trap-Neuter-Return community programs), while Dogs dominate Adoption and Return to Owner. Domestic Shorthair Mix represents $> 29\%$ of all intakes.
   - `SexUponIntake`: Neutered/Spayed animals are far more likely to be Return to Owner or Adoption; intact animals frequently represent strays that undergo transfer.
   - `time_in_shelter_days`: Adoption cases exhibit significantly higher stay durations (mean z-score $+0.3095$), whereas Transfers happen rapidly (mean z-score $-0.1832$).
3. **High Precision on 'Other':**  
   The `Other` class achieved an exceptional precision of **$88.69\%$**, primarily driven by specific animal types (e.g., bats and raccoons representing wildlife intake, which are never adopted or transferred).

---

### 6. Limitations & Future Work
1. **Feature Independence Violation:**  
   Breed and AnimalType are fundamentally correlated (e.g., Pit Bull Mix implies Dog). Treating them as conditionally independent double-counts breed-specific evidence.
2. **Gaussian Assumption on Skewed Durations:**  
   `time_in_shelter_days` and `age_days` follow heavy-tailed, exponential-like distributions rather than true Gaussian bell curves. Fitting a Gaussian distribution underestimates the probability mass of extreme short stays ($0$ days).
3. **Future Improvements:**  
   - Replace the Gaussian likelihood with a log-normal or Poisson distribution for stay duration.
   - Introduce intake condition and intake subtype interaction terms via semi-naive Bayesian structures.

---

### 7. Worked Example (Test Row $r = 95$)
For test sample index $r = 242013 \pmod{97} = 95$:
- **True Ground Truth:** `Transfer`
- **Feature Vector:**
  - `AnimalType = 0` (Cat) $\to \ln P(\text{Cat} \mid \text{Transfer}) = -0.5237$ vs $\ln P(\text{Cat} \mid \text{Return}) = -2.2699$
  - `SexUponIntake = 1` (Intact) $\to \ln P(\text{Intact} \mid \text{Transfer}) = -0.3761$
  - `Color = 2` (Black) $\to \ln P(\text{Black} \mid \text{Transfer}) = -1.3942$
  - `Breed = 0` (Domestic Shorthair Mix) $\to \ln P(\text{DSH} \mid \text{Transfer}) = -0.7755$
  - `age_days` (z-score = $-0.7164$) $\to$ Log-Likelihood = $+0.1654$
  - `time_in_shelter_days` (z-score = $-0.3963$) $\to$ Log-Likelihood = $+0.2195$
- **Class-wise Log-Posteriors:**
  - `Adoption`: $-6.9856$ ($P = 0.1822$)
  - `Transfer`: $\mathbf{-5.5267}$ ($P = \mathbf{0.7837}$)
  - `Return_to_Owner`: $-11.4212$ ($P = 0.0022$)
  - `Other`: $-8.7274$ ($P = 0.0319$)
- **Prediction:** The model correctly predicted **`Transfer`** with **$78.37\%$ posterior confidence**!
